# Figure 1c — transverse momentum and mass spectrumPanels a and b are schematics, composited in `figures/build_fig1.py`.Panel c is recorded on a **different spectrometer setting and a different set ofruns** from every other figure in the paper: 7594 V extraction on the 500 nmSiO2 sample, which is voltage setting 1 of the Supplementary Information. Allremaining figures use voltage setting 2. The x axis here is mass-over-charge,not flight time.Outputs: `figures/data/fig1c_x.npy`, `fig1c_histselected_.npy`,`fig1c_heatmap_data.npy`.

> **Data.** Reads the preprocessed per-run HDF5 files produced by> `scripts/preprocess.py`. Point `EUXFEL_NANO_DATA` at the directory holding> them. The raw data is at <https://doi.org/10.22003/XFEL.EU-DATA-003408-00>,> under embargo until publication.>> Figure 1 can be rebuilt **without** this data from the arrays already in> `figures/data/` — see `figures/README.md`.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.colors import LogNorm

from euxfel_nano import config
from euxfel_nano.io import data_root
from euxfel_nano.selection import ion_selection, fish_plot_ion_selection
from euxfel_nano.calibration import apply_calibration, mq_of

P = config.params()
print("data root:", data_root())

## Run selectionThe 7594 V runs, listed in `config/runs.yaml` under`sio2_500nm.runs.setting1_7594V`.

In [ ]:
RUNIDstretch = config.run_ids("500", transmission="setting1_7594V")
print(len(RUNIDstretch), "runs:", RUNIDstretch)

LOWER_BOUND = 1000
UPPER_BOUND = 1500
THRESHOLD = [(LOWER_BOUND, UPPER_BOUND)]

ion_dfevent_stretch, ion_dfpulse_stretch = ion_selection(RUNIDstretch, THRESHOLD)[0]

## Background for the calibration anchorsPulses with almost no ion signal contain only the argon carrier gas, whoseionization produces no recoil. Selecting the detector centre on these pulsesisolates the argon series used to anchor the mass calibration (Fig. S4).

In [ ]:
LOWER_BACKGRD_BOUND = 20
UPPER_BACKGRD_BOUND = 40
BKGRD_THRESHOLD = [(LOWER_BACKGRD_BOUND, UPPER_BACKGRD_BOUND)]
DOWNSAMPLING = 200000

backgrd_ion_dfevent, backgrd_ion_dfpulse = ion_selection(
    RUNIDstretch, BKGRD_THRESHOLD, DOWNSAMPLING)[0]

## Mass calibration — voltage setting 1The published panel applies the unconstrained power law `m/q = a * tof**b`fitted to the argon series. Its exponent b = 2.0175 is the value the SI reportsfor an unconstrained fit at this setting; the excess over 2 is the nonzero t0absorbed into the power law. The equivalent SI Eq. 1 form,`tof = t0 + k*sqrt(m/q)` with t0 = 7.451 ns and k = 2.0648e-7 s, agrees with itto about a nanosecond across the whole argon series — the cross-check belowprints the difference.

In [ ]:
cal = P["tof_calibration"]["voltage_setting_1"]
a_fit, b_fit = cal["legacy_power_law"]["a"], cal["legacy_power_law"]["b"]
print(f"m/q = {a_fit:.6e} * tof**{b_fit:.10f}")

ion_dfevent, backgrd_ion_dfevent_stretch = apply_calibration(
    [ion_dfevent_stretch, backgrd_ion_dfevent], a_fit, b_fit)

In [ ]:
# Cross-check: the two calibration forms agree to well under a nanosecond
# across the argon series at this setting.
for mq in P["tof_calibration"]["argon_lines_mq"]:
    t_pow = (mq / a_fit) ** (1 / b_fit)
    t_eq1 = cal["t0_s"] + cal["k_s"] * np.sqrt(mq)
    print(f"  m/q={mq:6.2f}   power law {t_pow*1e9:8.2f} ns   "
          f"Eq.1 {t_eq1*1e9:8.2f} ns   diff {abs(t_pow-t_eq1)*1e9:5.2f} ns")

## Panel cTop: the mass spectrum. Bottom: the same axis resolved along the detectorcoordinate that maps to transverse momentum, from a tilted cut through theinteraction region orthogonal to the X-ray beam (`fish_zone` in`config/params.yaml`).

In [ ]:
histselected, bin_edgesselected = np.histogram(
    ion_dfevent.mq, bins=np.linspace(0, 90, 1000), range=(0, 90))

Y_ZONE = tuple(P["spatial"]["fish_zone"])
y_lo, y_hi = P["spatial"]["fish_y_limits"]

fish_dfevent = fish_plot_ion_selection(ion_dfevent, Y_ZONE)
fish_dfevent_yfiltered = fish_dfevent[(fish_dfevent.y > y_lo) & (fish_dfevent.y < y_hi)]

mq_bins = 201
mq_bin_edges = np.linspace(0, 90, mq_bins)
fish_dfevent_yfiltered["mq_bin"] = pd.cut(
    fish_dfevent_yfiltered["mq"], bins=mq_bin_edges, labels=False)

heatmap_data = fish_dfevent_yfiltered.pivot_table(
    index="y", columns="mq_bin", aggfunc="size", fill_value=0)

max_value = heatmap_data.values.sum()
normalized_data = heatmap_data.values / max_value

In [ ]:
fig = plt.figure(figsize=(15, 8))
gs = gridspec.GridSpec(2, 1, height_ratios=[1, 1], hspace=0.15)
ax_main = plt.subplot(gs[0])
ax_heatmap = plt.subplot(gs[1], sharex=ax_main)

x = bin_edgesselected[:-1]
ax_main.plot(x, histselected / max(histselected), linewidth=2.5)
ax_main.set_ylabel("Norm. counts (arb. u.)", fontsize=20)
ax_main.tick_params(labelsize=16)
ax_main.set_ylim(0, 1.05)

cmap_magma = plt.get_cmap("magma").copy()
cmap_magma.set_bad(color="black")
masked_data = np.ma.masked_invalid(normalized_data)

center_val = 102
offsets = [-80, -40, 0, 40, 80]
ytick_vals = [center_val + offset for offset in offsets]
ytick_labels = [str(offset) for offset in offsets]

cax = ax_heatmap.imshow(
    masked_data,
    cmap=cmap_magma,
    aspect="auto",
    norm=LogNorm(vmin=3e-6, vmax=2e-4),
    extent=[x.min(), x.max(), 0, masked_data.shape[0]],
)

ax_heatmap.set_xlabel("m/q", fontsize=20)
ax_heatmap.set_yticks(ytick_vals)
ax_heatmap.set_yticklabels(ytick_labels, fontsize=16)
ax_heatmap.set_ylabel(r"$\propto\, p_\perp$", fontsize=20)
ax_heatmap.tick_params(labelsize=16)

cbar_ax = fig.add_axes([0.91, 0.115, 0.03, 0.35])
cbar = fig.colorbar(cax, cax=cbar_ax)
cbar.ax.set_ylabel("Norm. counts (arb. u.)", fontsize=20)
cbar.ax.tick_params(labelsize=16)

plt.show()

`figures/build_fig1.py` does its own normalization, so the raw pivot table iswhat gets saved, not `normalized_data`.

In [ ]:
np.save("../figures/data/fig1c_x.npy", x)
np.save("../figures/data/fig1c_histselected_.npy", histselected)
np.save("../figures/data/fig1c_heatmap_data.npy", heatmap_data)